# 03. Mô hình Image-Only & Vision-Centric cho ViMMSD

Notebook này thực hiện và đối sánh 3 bài toán trong nhóm **Image-Only / Vision-Centric**:
1. **Task 5 — Dùng CLIP Vision Encoder**: Fine-tune encoder đa phương thức tương phản (`openai/clip-vit-base-patch32`).
2. **Task 6 — Dùng ViT Image Encoder thuần**: Fine-tune Vision Transformer (`google/vit-base-patch16-224`) học trên ImageNet.
3. **Task 7 — Dùng VLM với Metadata**: Đánh giá năng lực của Vision-Language Model (Vintern-1B / Qwen2.5-VL) kết hợp OCR & mô tả ngữ cảnh hình ảnh.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config
from vimmsd.training.metrics import format_metrics

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

---
## 1. Task 5: Baseline Image-only dùng CLIP (openai/clip-vit-base-patch32)

* **Kiến trúc**: CLIP Vision Tower (`clip-vit-base-patch32`) + Classification Head (MLP 2 lớp).
* **Đặc điểm**: CLIP được pretrain tương phản (contrastive) giữa cặp Image-Text trên tập dữ liệu web khổng lồ, nên vector embedding mang ngữ nghĩa trừu tượng cao, nắm bắt tốt các biểu tượng và phong cách văn hóa.

In [ ]:
# Train Image-only CLIP
!python scripts/train.py --config configs/image_only.yaml --note "Task 5: CLIP Image-only"

In [ ]:
cfg_clip = load_config("configs/image_only.yaml")
out_clip = Path(cfg_clip.paths.output_dir) / cfg_clip.name
if (out_clip / "test_metrics.json").exists():
    metrics_clip = json.loads((out_clip / "test_metrics.json").read_text())
    print("=== KẾT QUẢ TASK 5 (CLIP IMAGE-ONLY) ===")
    print(format_metrics(metrics_clip))
    if (out_clip / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_clip / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có checkpoint tại {out_clip}. Hãy chạy cell train trước.")

---
## 2. Task 6: Baseline Image-only dùng ViT Image Encoder thuần (google/vit-base-patch16-224)

* **Kiến trúc**: ViT thuần (`vit-base-patch16-224`, pretrain trên ImageNet-21k / 1k) + Classification Head.
* **Mục tiêu so sánh**: ViT thuần chỉ học đặc trưng thị giác (visual features, textures, objects), hoàn toàn không có alignment với ngôn ngữ tự nhiên. Thí nghiệm này là ablation study quan trọng để chứng minh giá trị của multimodal contrastive pretraining (CLIP) so với supervised vision pretraining (ViT) trong bài toán mỉa mai.

In [ ]:
# Train Image-only ViT
!python scripts/train.py --config configs/image_only_vit.yaml --note "Task 6: ViT Image-only"

In [ ]:
cfg_vit = load_config("configs/image_only_vit.yaml")
out_vit = Path(cfg_vit.paths.output_dir) / cfg_vit.name
if (out_vit / "test_metrics.json").exists():
    metrics_vit = json.loads((out_vit / "test_metrics.json").read_text())
    print("=== KẾT QUẢ TASK 6 (ViT IMAGE-ONLY) ===")
    print(format_metrics(metrics_vit))
    if (out_vit / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_vit / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có checkpoint tại {out_vit}. Hãy chạy cell train trước.")

---
## 3. Task 7: Dùng VLM với Metadata (Visual Language Model)

* **Bản chất**: Ảnh mỉa mai trên mạng xã hội Việt Nam thường chứa **chữ trong ảnh (text in meme)** và **ngữ cảnh hành vi phức tạp** mà các visual encoder thông thường không nắm bắt được.
* **Cách tiếp cận**: Khai thác VLM (Vintern-1B hoặc Qwen2.5-VL) kết hợp metadata (chữ OCR trích xuất + mô tả ngữ cảnh chi tiết sinh bởi VLM) để nhận định sắc thái hình ảnh.

> **Task 7/10 (VLM + Metadata) — chưa hoàn thành trong PR này.**  
> Cần attach dataset `image_preprocessed.json` (output notebook `01b`, chứa OCR + mô tả VLM) làm input trên Kaggle trước khi chạy.  
> Sẽ tiếp tục ở PR sau khi có file cache.

In [ ]:
# Task 7: Đánh giá VLM (Vintern-1B) kết hợp metadata (OCR + mô tả ảnh)
# Yêu cầu: GPU >= 8 GB, dataset image_preprocessed.json đã attach trên Kaggle.
import torch
from vimmsd.data.dataset import load_all_records
from vimmsd.analysis.llm_compare import VinternClassifier, evaluate_llm

# Bật OCR + description từ cache — cần file image_preprocessed.json được attach
cfg_vlm = load_config("configs/base.yaml", [
    "data.image_text.use_ocr=true",
    "data.image_text.use_description=true",
])
test_records = load_all_records(cfg_vlm)["test"]
labels = list(cfg_vlm.data.labels)

out_vlm = Path(cfg_vlm.paths.output_dir) / "vlm_image_centric"
out_vlm.mkdir(parents=True, exist_ok=True)

vlm = VinternClassifier("5CD-AI/Vintern-1B-v3_5", labels=labels)

# Chế độ 1: zero-shot KHÔNG có metadata (baseline VLM thuần ảnh)
metrics_no_meta, _ = evaluate_llm(
    vlm, test_records, labels,
    use_metadata=False,
    out_path=out_vlm / "vintern_no_metadata.jsonl",  # out_path riêng
)
print("=== Vintern zero-shot (không metadata) ===")
print(format_metrics(metrics_no_meta))

# Chế độ 2: zero-shot CÓ metadata (OCR + mô tả ảnh)
metrics_meta, _ = evaluate_llm(
    vlm, test_records, labels,
    use_metadata=True,
    out_path=out_vlm / "vintern_with_metadata.jsonl",  # out_path riêng, khác chế độ 1
)
print("=== Vintern zero-shot (có OCR + mô tả ảnh) ===")
print(format_metrics(metrics_meta))

---
## 4. Tổng hợp & Đối sánh (Benchmark Table: CLIP vs ViT vs VLM)

Bảng so sánh hiệu năng của cả 3 phương pháp Image-Only / Vision-Centric.

In [ ]:
results = []
for name, p in [("CLIP (Task 5)", out_clip), ("ViT (Task 6)", out_vit)]:
    metric_file = p / "test_metrics.json"
    if metric_file.exists():
        m = json.loads(metric_file.read_text())
        results.append({
            "Phương pháp": name,
            "Macro-F1": m.get("macro_f1", 0),
            "Accuracy": m.get("accuracy", 0),
            **{f"F1 ({k})": v.get("f1", 0) for k, v in m.get("per_class", {}).items()}
        })

if results:
    df_res = pd.DataFrame(results).set_index("Phương pháp").round(4)
    display(df_res)
else:
    print("Chưa có kết quả đã train để lập bảng so sánh.")

---
## 5. Phân tích lỗi (Error Analysis)

So sánh các ca dự đoán sai để làm rõ: hình ảnh đơn lẻ thiếu thông tin gì so với khi có cả caption tiếng Việt?

In [ ]:
!python scripts/evaluate.py --config configs/image_only.yaml --split test
if (out_clip / "test_predictions.csv").exists():
    pred_clip = pd.read_csv(out_clip / "test_predictions.csv")
    wrong_clip = pred_clip[pred_clip.label != pred_clip.pred]
    print(f"CLIP Image-only sai: {len(wrong_clip)}/{len(pred_clip)} mẫu ({len(wrong_clip)/len(pred_clip)*100:.1f}%)")
    display(wrong_clip.groupby(["label", "pred"]).size().unstack(fill_value=0))
    print("10 mẫu sai ngẫu nhiên:")
    display(wrong_clip.sample(min(10, len(wrong_clip)), random_state=42)[["id", "label", "pred", "caption"]])